# Replication of Roud Song CLuters

Based on [Roud Song Clusters](https://github.com/sian0x0/Roud-Song-Clusters)
This uses our second dataset, dataset-v2.csv.
This version uses TF-IDF word scores and cosine distance instead of embeddings.

In [1]:
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.manifold import TSNE
import hdbscan

import plotly.express as px
import plotly.io as pio

import warnings
warnings.filterwarnings("ignore")

SEED = 1337  # for determinism

In [2]:
# Load the data, drop duplicates and any songs without lyrics
raw = pd.read_csv("dataset-v2.csv", dtype=str)

df_lyrics2 = pd.DataFrame({
    "name":    raw["title"],
    "roud":    raw["roud"],   # the answer we're evaluating
    "variant": raw["title"] + "# " + raw["version"],
    "lyrics":  raw["lyrics"],
})

df_lyrics2 = df_lyrics2.dropna(subset=["lyrics", "roud"])
df_lyrics2 = df_lyrics2.drop_duplicates(subset=["name", "lyrics", "roud"]).reset_index(drop=True)

print(len(df_lyrics2), "versions of", df_lyrics2.roud.nunique(), "songs")
df_lyrics2.head()

6828 versions of 2167 songs


,name,roud,variant,lyrics
0,A Jug of Punch / The 23 rd of June,1808,A Jug of Punch / The 23 rd of June# 1,"It being on the twenty-third of June-o,\nAs I ..."
1,A Jug of Punch / The 23 rd of June,1808,A Jug of Punch / The 23 rd of June# 2,As I was sitting with my glass and spoon\nOne ...
2,A Jug of Punch / The 23 rd of June,1808,A Jug of Punch / The 23 rd of June# 3,"It being on the third of June,\nas I sat weavi..."
3,A Jug of Punch / The 23 rd of June,1808,A Jug of Punch / The 23 rd of June# 4,As I was sitting with my jug and spoon\nOn one...
4,A Maid in Bedlam / Abroad As I Was Walking,578,A Maid in Bedlam / Abroad As I Was Walking# 1,Abroad as I was walking one evening in the spr...


In [3]:
# Keep ballads of three versions, since a cluster needs several members
df_lyrics2["roud_count"] = df_lyrics2.groupby("roud")["roud"].transform("count")
df_roud_multiples = df_lyrics2[df_lyrics2["roud_count"] >= 3]
print(len(df_roud_multiples), "variants of", df_roud_multiples.roud.nunique(), "ballads kept")

5023 variants of 887 ballads kept


In [4]:
# tf-idf step
df = df_lyrics2.copy()
lyrics = df["lyrics"].tolist()

vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True,
                             strip_accents="unicode", dtype=np.float32)
lyrics_tfidf = vectorizer.fit_transform(lyrics)

df_lyrics_w_tfidf = df.copy()
print("tf-idf matrix:", lyrics_tfidf.shape, "one row per song")

tf-idf matrix: (6828, 159417) one row per song


In [5]:
# cosine similarity: closer to 1.0 = same direction, lower = less alike
sim_table = lyrics_tfidf.dot(lyrics_tfidf.T).toarray()

# two versions of the same ballad vs. two different ballads
jug1, jug2 = df[df.roud == "1808"].index[:2]
fisherman  = df[df.roud == "291"].index[0]

print("same ballad     :", round(float(sim_table[jug1, jug2]), 3))
print("different ballad:", round(float(sim_table[jug1, fisherman]), 3))

# same check over the whole dataset: compare every pair of songs
roud = df.roud.to_numpy()
same_ballad = (roud.reshape(-1, 1) == roud.reshape(1, -1))
not_self = ~np.eye(len(df), dtype=bool)

print(f"\naverage similarity, same ballad      : {sim_table[same_ballad & not_self].mean():.3f}")
print(f"average similarity, different ballads: {sim_table[~same_ballad].mean():.3f}")

same ballad     : 0.159
different ballad: 0.023

average similarity, same ballad      : 0.286
average similarity, different ballads: 0.023


In [6]:
# check number of dimensions in the output
lyrics_tfidf.shape


(6828, 159417)

In [7]:
#t-sne, turning many dimensions into two 

df_multi_roud_w_tfidf = df_lyrics_w_tfidf[df_lyrics_w_tfidf.index.isin(df_roud_multiples.index)].copy()
X = lyrics_tfidf[df_multi_roud_w_tfidf.index.to_numpy()]

X_dist = np.clip(1.0 - X.dot(X.T).toarray(), 0.0, None)
X_dist = (X_dist + X_dist.T) / 2
np.fill_diagonal(X_dist, 0.0)

X_tsne = TSNE(n_components=2, metric="precomputed", init="random", random_state=SEED, n_jobs=-1).fit_transform(X_dist)
df_multi_roud_w_tfidf["tsne"] = X_tsne.tolist()
df_multi_roud_w_tfidf[["name", "roud", "tsne"]].head()

,name,roud,tsne
0,A Jug of Punch / The 23 rd of June,1808,"[8.235508, -46.394058]"
1,A Jug of Punch / The 23 rd of June,1808,"[9.077638, -45.850719]"
2,A Jug of Punch / The 23 rd of June,1808,"[8.365574, -46.288204]"
3,A Jug of Punch / The 23 rd of June,1808,"[8.566414, -46.1665]"
4,A Maid in Bedlam / Abroad As I Was Walking,578,"[40.406311, 35.461826]"


In [8]:
#hdbscan, turn datapoints into clusters
hdb_tsne = hdbscan.HDBSCAN(prediction_data=True, min_cluster_size=3, min_samples=2, gen_min_span_tree=True)
data = df_multi_roud_w_tfidf.tsne.to_list()
hdb_tsne.fit(data)

df_multi_roud_w_tfidf["cluster_tsne"] = hdb_tsne.labels_
df_multi_roud_w_tfidf["probability_tsne"] = hdb_tsne.probabilities_

labels = hdb_tsne.labels_
print("clusters found :", labels.max() + 1, " (true number of ballads:", df_multi_roud_w_tfidf.roud.nunique(), ")")
print(f"noise points   : {(labels == -1).sum()} ({(labels == -1).mean():.0%})")

clusters found : 1006  (true number of ballads: 887 )
noise points   : 174 (3%)



- **Pure cluster** = every song in it is the same ballad. This is good.
- **Mixed cluster** = contains 2+ different ballads, so the computer merged songs a human kept apart.
- **Split ballad** = one ballad spread over 2+ clusters, so the computer separated songs a human grouped.
- **Adjusted Rand Index (ARI):** 1.0 means perfect agreement with the human labels and 0 means random guessing.

In [9]:
# how good is the clustering?
from sklearn.metrics import adjusted_rand_score

clustered = df_multi_roud_w_tfidf[df_multi_roud_w_tfidf.cluster_tsne != -1]
per_cluster = clustered.groupby("cluster_tsne")["roud"].nunique()
per_ballad  = clustered.groupby("roud")["cluster_tsne"].nunique()

print(f"pure clusters : {(per_cluster == 1).sum()} / {len(per_cluster)}")
print(f"mixed clusters: {(per_cluster > 1).sum()}")
print(f"split ballads : {(per_ballad > 1).sum()} / {len(per_ballad)}")
# ARI: give each noise point its own singleton cluster. Otherwise sklearn treats all
# "-1" points as one big (wrong) cluster and the score is unfairly low.
lab = df_multi_roud_w_tfidf.cluster_tsne.to_numpy().copy()
noise = lab == -1
lab[noise] = lab.max() + 1 + np.arange(noise.sum())
print(f"ARI vs Child numbers: {adjusted_rand_score(df_multi_roud_w_tfidf.roud, lab):.2f}")

pure clusters : 888 / 1006
mixed clusters: 118
split ballads : 175 / 883
ARI vs Child numbers: 0.75


In [10]:
# chart/plot
df_plotly = df_multi_roud_w_tfidf.copy()
df_plotly["cluster_tsne"] = df_plotly["cluster_tsne"].astype(str)

# make non-cluster -1 grey
color_map = {"-1": "lightgray"}  # nb key is a string
# colours for other cluster labels
num_clusters = df_plotly["cluster_tsne"].nunique()
palette = px.colors.qualitative.Light24
color_map.update({str(i): palette[i % len(palette)] for i in range(num_clusters - 1)})

# calculate localized centroids for 'roud' annotations
def calculate_annotations():
    unique_roud_values = df_plotly["roud"].unique()
    annotations = []

    for roud_value in unique_roud_values:
        roud_clusters = df_plotly[df_plotly["roud"] == roud_value]["cluster_tsne"].unique()

        for cluster_label in roud_clusters:
            roud_points = df_plotly[(df_plotly["roud"] == roud_value) & (df_plotly["cluster_tsne"] == cluster_label)]
            if len(roud_points) > 0:
                centroid_x = roud_points["tsne"].str.get(0).mean()
                centroid_y = roud_points["tsne"].str.get(1).mean()
                if len(roud_points) > 1:
                    annotations.append(dict(x=centroid_x, y=centroid_y, xref="x", yref="y",
                                            text=roud_value, showarrow=True, arrowhead=2, ax=0, ay=-25))
                else:
                    annotations.append(dict(x=centroid_x, y=centroid_y, xref="x", yref="y",
                                            text=roud_value, showarrow=False, ax=0, ay=0))
    return annotations

annotations = calculate_annotations()

# plot
fig = px.scatter(df_plotly,
                 x=df_plotly["tsne"].str.get(0),
                 y=df_plotly["tsne"].str.get(1),
                 color="cluster_tsne",
                 color_discrete_map=color_map,
                 hover_name="name",
                 hover_data={"roud": True, "variant": True},
                 labels={"cluster_tsne": "Cluster Label", "roud": "Child no."},
                 title="Child ballads: t-SNE reduced HDBSCAN Clustering (labels = true Child number)")

button_on  = dict(label="Annotations On",  method="relayout", args=[{"annotations": annotations}])
button_off = dict(label="Annotations Off", method="relayout", args=[{"annotations": []}])
fig.update_layout(updatemenus=[dict(type="buttons", showactive=True, buttons=[button_on, button_off])],
                  annotations=annotations)

fig.update_layout(xaxis_title="t-SNE X", yaxis_title="t-SNE Y", showlegend=False)
fig.update_xaxes(scaleanchor="y", scaleratio=1)
fig.update_yaxes(scaleanchor="x", scaleratio=1)
fig.update_layout(width=1400, height=1100)

fig.write_html("ballad_clusters_tfidf.html")
fig.show()

fig.update_layout(updatemenus=[])
fig.write_image("ballad_clusters_tfidf.png", scale=2)
fig.show()


Each ballad's lyrics were turned into word scores (TF-IDF) over single words and word pairs, which gives high scores to the rare,
telling words a ballad shares across its versions. Cosine distance between those scores measures how alike two songs are. t-SNE
squashed those distances into 2 numbers so they can be drawn and so that dense neighborhoods stand out. HDBSCAN then circled the
dense neighbourhoods without being told how many to find, and it set aside the songs it couldn't place. Finally we compared the
circles with Child's human-made groups. About 88% of the clusters are **pure** (one ballad each), and the overall agreement
is decent but far from perfect with about a .76